# Uniformna raspodjela: bacanje kockice (knjiga, poglavlje 2.4.1)

$$p(x)=\begin{cases}\dfrac{1}{b-a}, & a\le x\le b\\ 0, & \text{inače}\end{cases}\qquad
P(x)=\begin{cases}0, & x<a\\ \dfrac{x-a}{b-a}, & a\le x\le b\\ 1, & x>b\end{cases}\qquad(2.6),(2.7)$$

**Eksperiment.** Kod poštene kockice nijedan broj nema prednost, pa svaki pada sa vjerovatnoćom $1/6$.
To je uniformna raspodjela u najčistijem obliku: „sve je jednako vjerovatno“.

Plan bilježnice:
1. obična kockica (diskretna uniformna raspodjela, programski kod 2.5),
2. kockica sa sve više strana: kada broj strana raste, stepenice funkcije raspodjele postaju prava linija (2.7), odnosno kontinualna uniformna raspodjela,
3. primjena: zadatak „Alma i Sara“ iz knjige (dolazak u slučajnom trenutku između 14 i 15 h).

**Pokretanje:** u Google Colabu `Runtime → Run all`. Prva ćelija sama klonira repozitorij
[ierovic1ba/ssp](https://github.com/ierovic1ba/ssp) (fork repozitorija [esokic/ssp](https://github.com/esokic/ssp) uz knjigu) i učita biblioteku `ssp`.
Svaka bilježnica radi $N=10\,000$ eksperimenata. Detaljna izvođenja su u PDF-u.

In [ ]:
# Priprema okruženja: kloniranje repozitorija sa SSP bibliotekom (prema README uputstvu)
import os, sys, subprocess

REPO_URL = "https://github.com/ierovic1ba/ssp.git"   # fork originala https://github.com/esokic/ssp
SSP_PUTANJA = "/content/ssp" if os.path.isdir("/content") else os.path.abspath("ssp")

if not os.path.isdir(os.path.join(SSP_PUTANJA, "ssp")):
    subprocess.run(["git", "clone", REPO_URL, SSP_PUTANJA], check=True)

sys.path.insert(0, SSP_PUTANJA)   # folder u kojem se nalazi paket ssp/
import ssp

import numpy as np
import matplotlib.pyplot as plt

# Podešavanje prikaza za rad studenata (preporuka iz README-a):
ssp.FORMATIRANJE_PRIKAZA = "JUPYTER"
ssp.REZOLUCIJA = "LOW"

N = 10000   # broj eksperimenata

## 1. Obična kockica
Očekivanje je $E[D]=\frac{1+2+\dots+6}{6}=3{,}5$, a varijansa $\mathrm{Var}[D]=\frac{91}{6}-3{,}5^2=\frac{35}{12}\approx2{,}917$.
Diskretna verzija formule $\frac{(b-a)^2}{12}$ glasi $\frac{n^2-1}{12}$, a za $n=6$ daje upravo $35/12$.

In [ ]:
kockica = np.random.randint(1, 7, size=N)        # N bacanja (programski kod 2.5)
frekv = np.array([np.mean(kockica == k) for k in range(1, 7)])

ssp.slprom1D.realizacije(np.arange(200), kockica[:200], naslov="Prvih 200 bacanja kockice")

plt.figure(figsize=(6, 3))
plt.bar(range(1, 7), frekv, 0.6, label="eksperiment")
plt.axhline(1/6, color="r", lw=2, label="teorija 1/6")
plt.title("Funkcija vjerovatnoće $p_X(x)$ za kockicu"); plt.xlabel("broj na kockici")
plt.ylim(0, 0.25); plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.show()

x, Px = ssp.slprom1D.PDF(kockica, step=0.01, prosirenje_opsega=0.15, prikaz=False)
ssp.slprom1D.grafik(x, Px, x, np.clip(np.floor(x), 0, 6)/6, stil2="r--",
                    x_labela="$x$", naslov="Funkcija raspodjele $P_X(x)$ (kao slika 2.3 u knjizi)",
                    legenda=["eksperiment", "teorija"], lokacija_lgd="lower right")

print("Relativne frekvencije:", np.round(frekv, 4))
print(f"E[D]   = {np.mean(kockica):.4f}   (teorija 3.5000)")
print(f"Var[D] = {np.var(kockica):.4f}   (teorija {35/12:.4f})")

## 2. Kockica sa sve više strana: od diskretne ka kontinualnoj
Zamislimo kockicu sa $n$ strana (u igrama postoje kockice sa 4, 6, 8, 12, 20 i 100 strana). Ako broj $k$ koji padne
pretvorimo u tačku $X=a+(b-a)\frac{k-1/2}{n}$ na intervalu $[a,b]$, svaka strana „pokriva“ dio intervala dužine $\frac{b-a}{n}$.
Za $n\to\infty$ stepenice funkcije raspodjele se stapaju u pravu liniju $\frac{x-a}{b-a}$, odnosno u relaciju (2.7).

In [ ]:
a, b = 0, 10                                     # kao u programskom kodu 2.1
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for osa, n_strana in zip(ax, [6, 20, 100, 10000]):
    k = np.random.randint(1, n_strana + 1, size=N)          # bacanje kockice sa n strana
    X = a + (b - a) * (k - 0.5) / n_strana
    osa.step(np.sort(X), np.arange(1, N + 1) / N, "b", where="post", label="eksperiment")
    osa.plot([a, b], [0, 1], "r--", label="(x-a)/(b-a)")
    osa.set_title(f"kockica sa {n_strana} strana"); osa.set_xlabel("$x$"); osa.grid(alpha=0.3)
ax[0].set_ylabel("$P(x)$"); ax[0].legend(loc="upper left")
fig.suptitle("Funkcija raspodjele: stepenice postaju prava linija"); fig.tight_layout(); plt.show()

In [ ]:
n_strana = 10000                                 # "kockica" sa jako mnogo strana = kontinualna uniformna
X = a + (b - a) * (np.random.randint(1, n_strana + 1, size=N) - 0.5) / n_strana
X_knjiga = np.random.uniform(a, b, N)            # programski kod 2.1

xs = np.linspace(a - 2, b + 2, 1000)
plt.figure(figsize=(6, 3.5))
plt.hist(X, bins=40, density=True, alpha=0.6, label=f"kockica sa {n_strana} strana")
plt.hist(X_knjiga, bins=40, density=True, histtype="step", lw=1.5, color="k", label="np.random.uniform")
plt.plot(xs, ((xs >= a) & (xs <= b)) / (b - a), "r", lw=2, label="teorija (2.6)")
plt.title("Funkcija gustine vjerovatnoće $p(x)$"); plt.xlabel("$x$")
plt.legend(loc="lower center"); plt.grid(alpha=0.3); plt.show()

print(f"E[X]   = {np.mean(X):.4f}  (teorija (a+b)/2    = {(a+b)/2:.4f})")
print(f"Var[X] = {np.var(X):.4f}  (teorija (b-a)^2/12 = {(b-a)**2/12:.4f})")

## 3. Primjena: Alma i Sara (knjiga, zadatak 1 iz poglavlja 3)
Alma i Sara dogovore sastanak u 14 h, a svaka stiže u slučajnom trenutku između 14 i 15 h (uniformno, nezavisno).
Ona koja stigne prva čeka 20 minuta. Kolika je vjerovatnoća da se sretnu?

Tačan odgovor dobija se geometrijski: na kvadratu $60\times60$ minuta „nepovoljna“ su dva trougla sa katetama 40,
pa je $P=1-\frac{40^2}{60^2}=\frac59\approx0{,}556$.

In [ ]:
alma = np.random.uniform(0, 60, N)               # minute nakon 14 h
sara = np.random.uniform(0, 60, N)
sretnu_se = np.abs(alma - sara) <= 20

plt.figure(figsize=(4.5, 4.5))
plt.plot(alma[~sretnu_se][:3000], sara[~sretnu_se][:3000], "r.", ms=2, label="ne sretnu se")
plt.plot(alma[sretnu_se][:3000], sara[sretnu_se][:3000], "g.", ms=2, label="sretnu se")
plt.xlabel("dolazak Alme [min]"); plt.ylabel("dolazak Sare [min]")
plt.gca().set_aspect("equal"); plt.legend(loc="upper left", framealpha=1); plt.show()

print(f"P(sretnu se) = {np.mean(sretnu_se):.4f}   (teorija 5/9 = {5/9:.4f})")